In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import json

# Load the dataset
dataset = load_dataset("socratesft/SocSci210") 

# Load a mapping file
mapping_file = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/participant_mapping.json",
    repo_type="dataset"
)

with open(mapping_file, 'r') as f:
    participant_mapping = json.load(f)

In [ ]:
import json

In [ ]:
from collections import defaultdict, Counter
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
len(dataset['train'])

In [ ]:
print(dataset['train'][0].keys())

In [ ]:
dataset['train'][1000]

In [ ]:
keys_val=defaultdict(list)

In [ ]:
for d in tqdm(dataset['train']):
    for k, v in d['demographic'].items():
        keys_val[k].append(v)

### Dataset Split

In [ ]:
study_id_list=[]
for d in tqdm(dataset['train']):
    study_id_list.append(d['study_id'])
study_id_list=list(set(study_id_list))
print(len(study_id_list))

In [ ]:
with open('./dataset/socsci/study_ids.json', 'w') as f:
    json.dump(study_id_list, f)

In [ ]:
import json

In [ ]:
save_data=[]
study_id='9nphm'
for d in tqdm(dataset['train']):
    if d['study_id']==study_id:
        save_data.append(d)
    else:
        with open('./dataset/socsci/socsci_{}.json'.format(study_id), 'w') as f:
            json.dump(save_data, f)
        print(f"Length of study id: {study_id}: {len(save_data)}")
        save_data=[]
        save_data.append(d)
        study_id=d['study_id']

In [ ]:
with open('./dataset/socsci/study_ids.json') as f:
    study_id_list=json.load(f)

In [ ]:
study_id_lens=[]
for study_id in tqdm(study_id_list):
    with open(f"./dataset/socsci/socsci_{study_id}.json") as f:
        data=json.load(f)
    study_id_lens.append((study_id, len(data)))

In [ ]:
study_id_lens[0]

In [ ]:
sorted_id=sorted(study_id_lens, key=lambda x: x[1], reverse=True)

In [ ]:
with open('./dataset/simbench/SimBenchGrouped_OpinionQA.json') as f:
    ex=json.load(f)

In [ ]:
ex[20]

In [ ]:
from tqdm import tqdm
from collections import defaultdict
kv_pair={}
interested_keys=['age', 'education', 'ethnicity', 'gender', 'ideology', 'income', 'location', 'marital_status', 'party_id']
for k in interested_keys:
    kv_pair[k]=[]
for i in tqdm(range(210)):
    with open(f'./dataset/socsci/socsci_{sorted_id[idx][0]}.json') as f:
        data=json.load(f)
    for d in (data):
        for k in interested_keys:
            if d['demographic'][k]!=None:
                kv_pair[k].append(d['demographic'][k])

    if all(len(v)!=0 for _, v in kv_pair.items()):
        break

In [ ]:
for k,v in kv_pair.items():
    kv_pair[k]=list(set(v))
len(kv_pair)

In [ ]:
kv_pair

In [ ]:
idx=0
with open(f'./dataset/socsci/socsci_{sorted_id[idx][0]}.json') as f:
    data=json.load(f)

In [ ]:
with open('./dataset/simbench/SimBenchGrouped_opinionqa.json') as f:
    opinion=json.load(f)

In [ ]:
data_name='mp3dr'
with open(f'./dataset/socsci/socsci_{data_name}.json') as f:
    data=json.load(f)

In [ ]:
print(f"Dataset Len: {len(data)}")
print(data[0]['stimuli'])
print(data[1]['stimuli'])

print(data[3]['stimuli'])
print(data[12]['stimuli'])

In [ ]:
"""
Dataset Grouping and Demographic Mapping Script
Maps survey data to standardized demographic categories and creates grouped distributions
"""

import pandas as pd
from collections import defaultdict, OrderedDict
import json
import re

# ============================================================================
# DEMOGRAPHIC MAPPING FUNCTIONS
# ============================================================================

def map_age_group(age):
    """
    Map age to age groups
    Target: '18-29', '30-49', '50-64', '65+'
    """
    if age is None:
        return None
    if 18 <= age <= 29:
        return '18-29'
    elif 30 <= age <= 49:
        return '30-49'
    elif 50 <= age <= 64:
        return '50-64'
    elif age >= 65:
        return '65+'
    return None

def map_location_to_region(location):
    """
    Map states to regions
    Target: 'South', 'Northeast', 'West', 'Midwest'
    """
    regions = {
        'West': ['Washington', 'Oregon', 'California', 'Nevada', 'Idaho', 'Montana', 
                 'Wyoming', 'Utah', 'Colorado', 'Arizona', 'New Mexico', 'Alaska', 'Hawaii'],
        'Northeast': ['Maine', 'New Hampshire', 'Vermont', 'Massachusetts', 'Rhode Island', 
                      'Connecticut', 'New York', 'New Jersey', 'Pennsylvania'],
        'South': ['Delaware', 'Maryland', 'Virginia', 'West Virginia', 'Kentucky', 'North Carolina',
                  'South Carolina', 'Tennessee', 'Georgia', 'Florida', 'Alabama', 'Mississippi',
                  'Arkansas', 'Louisiana', 'Oklahoma', 'Texas', 'District of Columbia'],
        'Midwest': ['North Dakota', 'South Dakota', 'Nebraska', 'Kansas', 'Minnesota', 'Iowa',
                    'Missouri', 'Wisconsin', 'Illinois', 'Michigan', 'Indiana', 'Ohio']
    }
    
    if location is None:
        return None
    
    for region, states in regions.items():
        if location in states:
            return region
    return None

def map_education(education):
    """
    Map education to standardized values
    Target: 'less than high school', 'high school graduate', 'some college, no degree',
            "associate's degree", 'college graduate/some postgrad', 'postgraduate'
    """
    if education is None:
        return None
    
    edu_str = str(education).strip()
    
    mapping = {
        # Less than high school
        'Less than high school': 'less than high school',
        'Some high school (no diploma)': 'less than high school',
        'No high school': 'less than high school',
        'Grade school': 'less than high school',
        'Some high school': 'less than high school',
        
        # High school graduate
        'High school graduate or equivalent': 'high school graduate',
        'High school graduate': 'high school graduate',
        'High school diploma': 'high school graduate',
        'GED': 'high school graduate',
        'High school grad': 'high school graduate',
        
        # Some college, no degree
        'Some college': 'some college, no degree',
        'Some college, no degree': 'some college, no degree',
        'Vocational/tech school': 'some college, no degree',
        'Vocational/tech school/some college/associates': 'some college, no degree',
        
        # Associate's degree
        "Associate's degree": "associate's degree",
        'Associate degree': "associate's degree",
        'Associates degree': "associate's degree",
        'AA': "associate's degree",
        'AS': "associate's degree",
        
        # College graduate/some postgrad
        "Bachelor's degree": 'college graduate/some postgrad',
        'Bachelor degree': 'college graduate/some postgrad',
        'Bachelors degree': 'college graduate/some postgrad',
        'BA': 'college graduate/some postgrad',
        'BS': 'college graduate/some postgrad',
        'College graduate': 'college graduate/some postgrad',
        'Four-year college graduate': 'college graduate/some postgrad',
        
        # Postgraduate
        'Post grad study/professional degree': 'postgraduate',
        'Postgraduate': 'postgraduate',
        'Graduate degree': 'postgraduate',
        'Masters degree': 'postgraduate',
        "Master's degree": 'postgraduate',
        'Doctoral degree': 'postgraduate',
        'PhD': 'postgraduate',
        'JD': 'postgraduate',
        'MD': 'postgraduate',
        'Professional degree': 'postgraduate',
        'Graduate or professional degree': 'postgraduate'
    }
    
    return mapping.get(edu_str, None)

def map_marital_status(marital_status):
    """
    Map marital status to standardized values
    Target: 'married', 'divorced', 'widowed', 'living with a partner', 'never been married'
    """
    if marital_status is None:
        return None
    
    marital_str = str(marital_status).strip()
    
    mapping = {
        # Married
        'Married': 'married',
        'Married, spouse present': 'married',
        'Married, spouse absent': 'married',
        
        # Never married
        'Never married': 'never been married',
        'Never been married': 'never been married',
        'Single': 'never been married',
        'Single, never married': 'never been married',
        
        # Divorced
        'Divorced': 'divorced',
        'Separated': 'divorced',
        
        # Widowed
        'Widowed': 'widowed',
        'Widow': 'widowed',
        'Widower': 'widowed',
        
        # Living with partner
        'Living with partner': 'living with a partner',
        'Living with a partner': 'living with a partner',
        'Domestic partnership': 'living with a partner',
        'Cohabiting': 'living with a partner',
        'Unmarried couple': 'living with a partner'
    }
    
    return mapping.get(marital_str, None)

def map_ideology(ideology):
    """
    Map ideology to standardized values
    Target: 'very liberal', 'liberal', 'moderate', 'conservative', 'very conservative'
    """
    if ideology is None:
        return None
    
    ideology_str = str(ideology).strip()
    
    mapping = {
        # Very liberal
        'Extremely Liberal': 'very liberal',
        'Very Liberal': 'very liberal',
        'Very liberal': 'very liberal',
        
        # Liberal
        'Liberal': 'liberal',
        'Somewhat Liberal': 'liberal',
        'Slightly liberal': 'liberal',
        'Lean liberal': 'liberal',
        
        # Moderate
        'Moderate': 'moderate',
        'Middle of the road': 'moderate',
        
        # Conservative
        'Conservative': 'conservative',
        'Somewhat Conservative': 'conservative',
        'Slightly conservative': 'conservative',
        'Lean conservative': 'conservative',
        
        # Very conservative
        'Extremely Conservative': 'very conservative',
        'Very Conservative': 'very conservative',
        'Very conservative': 'very conservative',
        
        # Not applicable
        'Declined to Answer': None,
        "Don't know": None,
        'Refused': None,
        'Not sure': None
    }
    
    return mapping.get(ideology_str, None)

def map_income(income):
    """
    Map income to standardized ranges
    Target: 'Less than $30,000', '$30,000-$50,000', '$50,000-$75,000',
            '$75,000-$100,000', '$100,000 or more'
    """
    if income is None:
        return None
    
    income_str = str(income).strip()
    
    mapping = {
        # Less than $30,000
        'Less than $10,000': 'Less than $30,000',
        '$10,000-$19,999': 'Less than $30,000',
        '10-19K': 'Less than $30,000',
        '20-29K': 'Less than $30,000',
        '$20,000-$29,999': 'Less than $30,000',
        'Under $30,000': 'Less than $30,000',
        'Less than $30,000': 'Less than $30,000',
        
        # $30,000-$50,000
        '30-39K': '$30,000-$50,000',
        '40-49K': '$30,000-$50,000',
        '$30,000-$39,999': '$30,000-$50,000',
        '$40,000-$49,999': '$30,000-$50,000',
        '$30,000-$50,000': '$30,000-$50,000',
        
        # $50,000-$75,000
        '50-74K': '$50,000-$75,000',
        '$50,000-$74,999': '$50,000-$75,000',
        '$50,000-$75,000': '$50,000-$75,000',
        
        # $75,000-$100,000
        '75-99K': '$75,000-$100,000',
        '$75,000-$99,999': '$75,000-$100,000',
        '$75,000-$100,000': '$75,000-$100,000',
        
        # $100,000 or more
        '100-124K': '$100,000 or more',
        '125-149K': '$100,000 or more',
        '150-175K+': '$100,000 or more',
        '175-200K+': '$100,000 or more',
        '200K+': '$100,000 or more',
        '$100,000-$124,999': '$100,000 or more',
        '$125,000-$149,999': '$100,000 or more',
        '$150,000-$174,999': '$100,000 or more',
        '$175,000-$199,999': '$100,000 or more',
        '$200,000 or more': '$100,000 or more',
        '$100,000 or more': '$100,000 or more',
        '$100,000+': '$100,000 or more',
        '100K+': '$100,000 or more'
    }
    
    return mapping.get(income_str, None)

def map_gender(gender):
    """
    Map gender to standardized values
    Target: 'male', 'female'
    """
    if gender is None:
        return None
    
    gender_str = str(gender).strip()
    
    mapping = {
        # Male variations
        'Male': 'male',
        'male': 'male',
        'man': 'male',
        'Man': 'male',
        'm': 'male',
        'M': 'male',
        
        # Female variations
        'Female': 'female',
        'female': 'female',
        'woman': 'female',
        'Woman': 'female',
        'f': 'female',
        'F': 'female'
    }
    
    return mapping.get(gender_str, None)

def map_ethnicity_to_race(ethnicity):
    """
    Map ethnicity to race categories
    Target: 'white', 'black', 'hispanic', 'asian'
    """
    if ethnicity is None:
        return None
    
    ethnicity_str = str(ethnicity).strip()
    
    mapping = {
        # White
        'White': 'white',
        'White, non-Hispanic': 'white',
        'Caucasian': 'white',
        'Non-Hispanic White': 'white',
        
        # Black
        'Black': 'black',
        'Black or African American': 'black',
        'African American': 'black',
        'Non-Hispanic Black': 'black',
        
        # Hispanic
        'Hispanic': 'hispanic',
        'Latino': 'hispanic',
        'Latina': 'hispanic',
        'Latinx': 'hispanic',
        'Hispanic or Latino': 'hispanic',
        'Mexican': 'hispanic',
        'Puerto Rican': 'hispanic',
        'Cuban': 'hispanic',
        
        # Asian
        'Asian': 'asian',
        'Asian or Pacific Islander': 'asian',
        'Asian American': 'asian',
        'Chinese': 'asian',
        'Japanese': 'asian',
        'Korean': 'asian',
        'Filipino': 'asian',
        'Vietnamese': 'asian',
        'Indian': 'asian',
        'South Asian': 'asian',
        'East Asian': 'asian',
        'Southeast Asian': 'asian',
        
        # Not in target categories - return None
        'Native American': None,
        'American Indian': None,
        'Alaska Native': None,
        'Native Hawaiian': None,
        'Pacific Islander': None,
        'Other': None,
        'Two or more races': None,
        'Multiracial': None,
        'Mixed': None
    }
    
    return mapping.get(ethnicity_str, None)

def map_party_id(party_id):
    """
    Map party identification to standardized values
    Target: 'Democrat', 'Republican', 'Independent', 'Other'
    """
    if party_id is None:
        return None
    
    party_str = str(party_id).strip()
    
    mapping = {
        # Democrat variations
        'Democrat': 'Democrat',
        'Democratic': 'Democrat',
        'Strong Democrat': 'Democrat',
        'Moderate Democrat': 'Democrat',
        'Lean Democrat': 'Democrat',
        'Lean Democratic': 'Democrat',
        'Weak Democrat': 'Democrat',
        'Not very strong Democrat': 'Democrat',
        
        # Republican variations
        'Republican': 'Republican',
        'GOP': 'Republican',
        'Strong Republican': 'Republican',
        'Moderate Republican': 'Republican',
        'Lean Republican': 'Republican',
        'Lean GOP': 'Republican',
        'Weak Republican': 'Republican',
        'Not very strong Republican': 'Republican',
        
        # Independent variations
        'Independent': 'Independent',
        'Ind': 'Independent',
        "Don't Lean/Independent/None": 'Independent',
        'Independent (No party)': 'Independent',
        'No preference': 'Independent',
        'Unaffiliated': 'Independent',
        
        # Other
        'Other': 'Other',
        'Other party': 'Other',
        'Third party': 'Other',
        'Minor party': 'Other',
        'Libertarian': 'Other',
        'Green': 'Other',
        'Green Party': 'Other',
        'Libertarian Party': 'Other'
    }
    
    return mapping.get(party_str, None)

# Feature name mapping to standard keys
FEATURE_TO_KEY_MAP = {
    'region': 'CREGION',
    'age': 'AGE',
    'gender': 'SEX',
    'education': 'EDUCATION',
    'marital_status': 'MARITAL',
    'ideology': 'POLIDEOLOGY',
    'income': 'INCOME',
    'party_id': 'POLPARTY',
    'ethnicity': 'RACE'
}

# ============================================================================
# ANSWER OPTION EXTRACTION
# ============================================================================

def extract_answer_options(stimuli, responses):
    """
    Extract ALL possible answer options from stimuli text
    Returns dict mapping option keys to labels
    """
    answer_options = {}
    
    # Check for Yes/No questions
    if "1 for Yes or 2 for No" in stimuli or "1 for Yes and 2 for No" in stimuli:
        answer_options = {
            '1': 'Yes',
            '2': 'No'
        }
    # Check for letter options (A), (B), (C), etc.
    elif "(A):" in stimuli or "(A)" in stimuli:
        lines = stimuli.split('\n')
        for line in lines:
            for letter in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']:
                if f'({letter}):' in line or f'({letter})' in line:
                    if f'({letter}):' in line:
                        parts = line.split(f'({letter}):', 1)
                    elif f'({letter})' in line:
                        parts = line.split(f'({letter})', 1)
                    else:
                        continue
                    
                    if len(parts) == 2:
                        answer_text = parts[1].strip()
                        answer_text = re.sub(r'\s*\([A-Z]\).*$', '', answer_text)
                        answer_options[letter] = answer_text
    # Check for numeric scale patterns
    else:
        scale_patterns = [
            r'from\s+(\d+)\s+to\s+(\d+)',
            r'between\s+(\d+)\s+and\s+(\d+)',
            r'integer\s+from\s+(\d+)\s+to\s+(\d+)',
            r'scale\s+of\s+(\d+)\s+to\s+(\d+)',
            r'(\d+)\s+to\s+(\d+)',
        ]
        
        min_val = None
        max_val = None
        
        for pattern in scale_patterns:
            match = re.search(pattern, stimuli, re.IGNORECASE)
            if match:
                min_val = int(match.group(1))
                max_val = int(match.group(2))
                break
        
        if min_val is not None and max_val is not None:
            # Try to find labels for min and max values
            label_patterns_min = [
                r'where\s+' + str(min_val) + r'\s+means?\s+["\']([^"\']+)["\']',
                r'where\s+' + str(min_val) + r'\s+means?\s+([^,\.]+?)(?:\s+and|\s+or|\s*,)',
                str(min_val) + r'\s*[=:]\s*["\']?([^"\',:\.]+)["\']?',
            ]
            
            min_label = None
            max_label = None
            
            for pattern in label_patterns_min:
                match = re.search(pattern, stimuli, re.IGNORECASE)
                if match:
                    min_label = match.group(1).strip()
                    break
            
            label_patterns_max = [
                r'where\s+' + str(max_val) + r'\s+means?\s+["\']([^"\']+)["\']',
                r'and\s+' + str(max_val) + r'\s+means?\s+["\']([^"\']+)["\']',
                r'and\s+' + str(max_val) + r'\s+means?\s+([^,\.]+?)(?:\s*,|\s*\.|\s*\')',
                str(max_val) + r'\s*[=:]\s*["\']?([^"\',:\.]+)["\']?',
            ]
            
            for pattern in label_patterns_max:
                match = re.search(pattern, stimuli, re.IGNORECASE)
                if match:
                    max_label = match.group(1).strip()
                    break
            
            # Create answer options for all values in range
            for val in range(min_val, max_val + 1):
                if val == min_val and min_label:
                    answer_options[str(val)] = min_label
                elif val == max_val and max_label:
                    answer_options[str(val)] = max_label
                else:
                    answer_options[str(val)] = f'Response {val}'
        else:
            # Fallback: use unique responses
            unique_responses = set(responses)
            answer_options = {str(resp): f"Response {resp}" for resp in sorted(unique_responses)}
    
    return answer_options

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def create_group_prompt(feature, value):
    """Create a natural language prompt for the group"""
    prompts = {
        'age': f'You are in the age group {value}.',
        'region': f'You are from the {value} region.',
        'education': f'Your education level is: {value}.',
        'gender': f'You are {value}.',
        'ideology': f'Your political ideology is: {value}.',
        'income': f'Your household income is: {value}.',
        'marital_status': f'Your marital status is: {value}.',
        'party_id': f'Your party identification is: {value}.',
        'ethnicity': f'Your race is: {value}.'
    }
    
    return prompts.get(feature, f'{feature}: {value}')

def create_human_answer(response_counts, answer_options, total_responses):
    """
    Create human_answer distribution with ALL possible options
    Include 0.0 for options that weren't selected
    """
    human_answer = OrderedDict()
    
    # Get all possible keys from answer_options and sort
    all_keys = sorted(answer_options.keys(), key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else 0, x))
    
    for key in all_keys:
        count = 0
        # Try both string and int versions of the key
        if key in response_counts:
            count = response_counts[key]
        elif key.isdigit() and int(key) in response_counts:
            count = response_counts[int(key)]
        
        percentage = (count / total_responses * 100) if total_responses > 0 else 0.0
        human_answer[key] = percentage
    
    return human_answer

# ============================================================================
# MAIN PROCESSING FUNCTIONS
# ============================================================================

def process_dataset(dataset, demographic_features, min_group_size=None, max_group_size=None):
    """
    Process dataset and group by question (stimuli) AND demographic features
    
    Args:
        dataset: List of dictionaries with your data structure
        demographic_features: List of demographic features to group by
        min_group_size: Minimum number of respondents required in a group
        max_group_size: Maximum number of respondents allowed in a group
    
    Returns:
        Tuple of (results, filtered_out)
    """
    
    # Process each instance and add mapped values
    processed_data = []
    for instance in dataset:
        demo = instance['demographic']
        
        processed_instance = {
            'sample_id': instance['sample_id'],
            'participant': instance['participant'],
            'stimuli': instance['stimuli'],
            'response': instance['response'],
            'prompt': instance['prompt'],
            'study_id': instance.get('study_id'),
            'task_num': instance.get('task_num'),
            'condition_num': instance.get('condition_num'),
            'demographics': {
                'age': demo.get('age'),
                'age_group': map_age_group(demo.get('age')),
                'education': map_education(demo.get('education')),
                'employment': demo.get('employment'),
                'ethnicity': map_ethnicity_to_race(demo.get('ethnicity')),
                'gender': map_gender(demo.get('gender')),
                'ideology': map_ideology(demo.get('ideology')),
                'income': map_income(demo.get('income')),
                'location': demo.get('location'),
                'region': map_location_to_region(demo.get('location')),
                'marital_status': map_marital_status(demo.get('marital_status')),
                'party_id': map_party_id(demo.get('party_id'))
            }
        }
        processed_data.append(processed_instance)
    
    # Group by stimuli first
    questions = defaultdict(list)
    for instance in processed_data:
        questions[instance['stimuli']].append(instance)
    
    print(f"Found {len(questions)} unique questions based on stimuli")
    
    overall_results = []
    demographic_results = []
    filtered_out = []
    
    question_index = 0

    # For each unique question
    for stimuli, question_instances in questions.items():
        sample_instance = question_instances[0]
        all_responses = [inst['response'] for inst in question_instances]
        
        # Extract ALL possible answer options from the question text
        answer_options = extract_answer_options(stimuli, all_responses)
        output_format = 'choice' if answer_options else 'numeric'
        
        # Create overall distribution
        overall_response_counts = defaultdict(int)
        total_overall = len(question_instances)
        
        for inst in question_instances:
            overall_response_counts[inst['response']] += 1
        
        overall_human_answer = create_human_answer(overall_response_counts, answer_options, total_overall)
        
        overall_result = {
            'question_idx': question_index,
            'group_prompt_template': 'You are from the United States.',
            'group_prompt_variable_map': {},
            'input_template': stimuli,
            'input_variable_map': {},
            'output_format': output_format,
            'answer_options': answer_options,
            'auxiliary': {
                'task_num': sample_instance.get('task_num'),
                'condition_num': sample_instance.get('condition_num')
            },
            'group_size': total_overall,
            'human_answer': dict(overall_human_answer),
            'num_grouping_vars': 0,
            'dataset_name': sample_instance.get('study_id', 'Unknown')
        }
        
        overall_results.append(overall_result)
        
        # Create demographic group breakdowns
        for feature in demographic_features:
            # Determine which field to use
            if feature == 'age':
                group_field = 'age_group'
            elif feature == 'location':
                group_field = 'region'
            else:
                group_field = feature
            
            # Get standardized key name
            standard_key = FEATURE_TO_KEY_MAP.get(feature, feature.upper())
            
            groups = defaultdict(list)
            for instance in question_instances:
                group_value = instance['demographics'].get(group_field)
                if group_value is not None:
                    groups[group_value].append(instance)
            
            # Create output for each group
            for group_value, instances in groups.items():
                response_counts = defaultdict(int)
                total_responses = len(instances)
                
                filter_reason = None
                if min_group_size is not None and total_responses < min_group_size:
                    filter_reason = f"too_small (size={total_responses}, min={min_group_size})"
                elif max_group_size is not None and total_responses > max_group_size:
                    filter_reason = f"too_large (size={total_responses}, max={max_group_size})"
                
                for inst in instances:
                    response_counts[inst['response']] += 1
                
                human_answer = create_human_answer(response_counts, answer_options, total_responses)
                
                # Create prompt with proper format
                prompt_text = create_group_prompt(feature, group_value)
                group_prompt_template = f'You are from the United States. {prompt_text}'
                
                result = {
                    'question_idx': question_index,
                    'group_prompt_template': group_prompt_template,
                    'group_prompt_variable_map': {standard_key: group_value},
                    'input_template': stimuli,
                    'input_variable_map': {},
                    'output_format': output_format,
                    'answer_options': answer_options,
                    'auxiliary': {
                        'task_num': sample_instance.get('task_num'),
                        'condition_num': sample_instance.get('condition_num')
                    },
                    'group_size': total_responses,
                    'human_answer': dict(human_answer),
                    'num_grouping_vars': 1,
                    'dataset_name': sample_instance.get('study_id', 'Unknown')
                }
                
                if filter_reason:
                    result['filter_reason'] = filter_reason
                    filtered_out.append(result)
                else:
                    demographic_results.append(result)
        
        question_index += 1
    
    results = overall_results + demographic_results
    
    print(f"\nResults Statistics:")
    print(f"  Overall distributions: {len(overall_results)}")
    print(f"  Demographic groups kept: {len(demographic_results)}")
    print(f"  Demographic groups filtered out: {len(filtered_out)}")
    print(f"  Total results: {len(results)}")
    
    return results, filtered_out

def process_dataset_multi_group(dataset, grouping_combinations, min_group_size=None, max_group_size=None, 
                                include_overall=True):
    """
    Group by multiple demographic features simultaneously for each question
    
    Args:
        dataset: Your dataset
        grouping_combinations: List of tuples, e.g., [('age', 'gender'), ('region', 'party_id')]
        min_group_size: Minimum number of respondents required in a group
        max_group_size: Maximum number of respondents allowed in a group
        include_overall: Whether to include overall distribution for each question
    
    Returns:
        Tuple of (results, filtered_out)
    """
    # Process instances
    processed_data = []
    for instance in dataset:
        demo = instance['demographic']
        processed_instance = {
            'sample_id': instance['sample_id'],
            'response': instance['response'],
            'stimuli': instance['stimuli'],
            'study_id': instance.get('study_id'),
            'task_num': instance.get('task_num'),
            'condition_num': instance.get('condition_num'),
            'demographics': {
                'age_group': map_age_group(demo.get('age')),
                'education': map_education(demo.get('education')),
                'employment': demo.get('employment'),
                'ethnicity': map_ethnicity_to_race(demo.get('ethnicity')),
                'gender': map_gender(demo.get('gender')),
                'ideology': map_ideology(demo.get('ideology')),
                'income': map_income(demo.get('income')),
                'region': map_location_to_region(demo.get('location')),
                'marital_status': map_marital_status(demo.get('marital_status')),
                'party_id': map_party_id(demo.get('party_id'))
            }
        }
        processed_data.append(processed_instance)
    
    questions = defaultdict(list)
    for instance in processed_data:
        questions[instance['stimuli']].append(instance)
    
    print(f"Found {len(questions)} unique questions based on stimuli")
    
    overall_results = []
    demographic_results = []
    filtered_out = []
    
    question_index = 0
    
    for stimuli, question_instances in questions.items():
        sample_instance = question_instances[0]
        all_responses = [inst['response'] for inst in question_instances]
        
        answer_options = extract_answer_options(stimuli, all_responses)
        output_format = 'choice' if answer_options else 'numeric'
        
        # Overall distribution
        if include_overall:
            overall_response_counts = defaultdict(int)
            total_overall = len(question_instances)
            
            for inst in question_instances:
                overall_response_counts[inst['response']] += 1
            
            overall_human_answer = create_human_answer(overall_response_counts, answer_options, total_overall)
            
            overall_result = {
                'question_idx': question_index,
                'group_prompt_template': 'You are from the United States.',
                'group_prompt_variable_map': {},
                'input_template': stimuli,
                'input_variable_map': {},
                'output_format': output_format,
                'answer_options': answer_options,
                'auxiliary': {
                    'task_num': sample_instance.get('task_num'),
                    'condition_num': sample_instance.get('condition_num')
                },
                'group_size': total_overall,
                'human_answer': dict(overall_human_answer),
                'num_grouping_vars': 0,
                'dataset_name': sample_instance.get('study_id', 'Unknown')
            }
            
            overall_results.append(overall_result)
        
        # Multi-demographic group breakdowns
        for combo in grouping_combinations:
            groups = defaultdict(list)
            
            for instance in question_instances:
                group_key = tuple(
                    instance['demographics'].get(
                        'age_group' if feat == 'age' else 'region' if feat == 'location' else feat
                    )
                    for feat in combo
                )
                
                if None not in group_key:
                    groups[group_key].append(instance)
            
            for group_key, instances in groups.items():
                response_counts = defaultdict(int)
                for inst in instances:
                    response_counts[inst['response']] += 1
                
                total = len(instances)
                
                filter_reason = None
                if min_group_size is not None and total < min_group_size:
                    filter_reason = f"too_small (size={total}, min={min_group_size})"
                elif max_group_size is not None and total > max_group_size:
                    filter_reason = f"too_large (size={total}, max={max_group_size})"
                
                human_answer = create_human_answer(response_counts, answer_options, total)
                
                # Create variable map with standardized keys
                variable_map = {
                    FEATURE_TO_KEY_MAP.get(feat, feat.upper()): val 
                    for feat, val in zip(combo, group_key)
                }
                
                # Create combined prompt
                prompt_parts = []
                for feat, val in zip(combo, group_key):
                    prompt_parts.append(create_group_prompt(feat, val))
                
                group_prompt_template = f"You are from the United States. {' '.join(prompt_parts)}"
                
                result = {
                    'question_idx': question_index,
                    'group_prompt_template': group_prompt_template,
                    'group_prompt_variable_map': variable_map,
                    'input_template': stimuli,
                    'input_variable_map': {},
                    'output_format': output_format,
                    'answer_options': answer_options,
                    'auxiliary': {
                        'task_num': sample_instance.get('task_num'),
                        'condition_num': sample_instance.get('condition_num')
                    },
                    'group_size': total,
                    'human_answer': dict(human_answer),
                    'num_grouping_vars': len(combo),
                    'dataset_name': sample_instance.get('study_id', 'Unknown')
                }
                
                if filter_reason:
                    result['filter_reason'] = filter_reason
                    filtered_out.append(result)
                else:
                    demographic_results.append(result)
        
        question_index += 1
    
    results = overall_results + demographic_results
    
    print(f"\nResults Statistics:")
    if include_overall:
        print(f"  Overall distributions: {len(overall_results)}")
    print(f"  Multi-demographic groups kept: {len(demographic_results)}")
    print(f"  Multi-demographic groups filtered out: {len(filtered_out)}")
    print(f"  Total results: {len(results)}")
    
    return results, filtered_out

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def save_results(results, output_file='grouped_results.json'):
    """Save results to JSON file"""
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {output_file}")

def save_filtered_out(filtered_out, output_file='filtered_out_groups.json'):
    """Save filtered out groups to JSON file"""
    with open(output_file, 'w') as f:
        json.dump(filtered_out, f, indent=2)
    print(f"Filtered out groups saved to {output_file}")

def print_unmapped_values(dataset):
    """Print all unique values that couldn't be mapped"""
    unmapped = {
        'party_id': set(),
        'education': set(),
        'marital_status': set(),
        'ideology': set(),
        'income': set(),
        'gender': set(),
        'ethnicity': set(),
        'location': set()
    }
    
    for instance in dataset:
        demo = instance['demographic']
        
        party = demo.get('party_id')
        if party is not None and map_party_id(party) is None:
            unmapped['party_id'].add(party)
        
        edu = demo.get('education')
        if edu is not None and map_education(edu) is None:
            unmapped['education'].add(edu)
        
        marital = demo.get('marital_status')
        if marital is not None and map_marital_status(marital) is None:
            unmapped['marital_status'].add(marital)
        
        ideology = demo.get('ideology')
        if ideology is not None and map_ideology(ideology) is None:
            unmapped['ideology'].add(ideology)
        
        income = demo.get('income')
        if income is not None and map_income(income) is None:
            unmapped['income'].add(income)
        
        gender = demo.get('gender')
        if gender is not None and map_gender(gender) is None:
            unmapped['gender'].add(gender)
        
        ethnicity = demo.get('ethnicity')
        if ethnicity is not None and map_ethnicity_to_race(ethnicity) is None:
            unmapped['ethnicity'].add(ethnicity)
        
        location = demo.get('location')
        if location is not None and map_location_to_region(location) is None:
            unmapped['location'].add(location)
    
    print(f"\n{'='*60}")
    print("UNMAPPED VALUES REPORT")
    print(f"{'='*60}")
    
    has_unmapped = False
    for field, values in unmapped.items():
        if values:
            has_unmapped = True
            print(f"\n{field.upper()}:")
            for val in sorted(values):
                print(f"  - '{val}'")
    
    if not has_unmapped:
        print("\n✓ All values successfully mapped!")
    else:
        print(f"\n⚠️  Found unmapped values - add these to mapping functions")
    
    return unmapped

def verify_mapping_values(dataset):
    """Verify that all mapped values match the target format"""
    target_values = {
        'CREGION': {'South', 'Northeast', 'West', 'Midwest'},
        'AGE': {'65+', '18-29', '50-64', '30-49'},
        'SEX': {'male', 'female'},
        'EDUCATION': {'high school graduate', 'college graduate/some postgrad', 
                      'some college, no degree', "associate's degree", 'postgraduate', 
                      'less than high school'},
        'MARITAL': {'married', 'divorced', 'widowed', 'living with a partner', 'never been married'},
        'INCOME': {'$30,000-$50,000', '$75,000-$100,000', '$100,000 or more', 
                   '$50,000-$75,000', 'Less than $30,000'},
        'POLIDEOLOGY': {'liberal', 'moderate', 'conservative', 'very conservative', 'very liberal'},
        'POLPARTY': {'Democrat', 'Independent', 'Other', 'Republican'},
        'RACE': {'black', 'hispanic', 'asian', 'white'}
    }
    
    field_map = {
        'region': 'CREGION',
        'age_group': 'AGE',
        'gender': 'SEX',
        'education': 'EDUCATION',
        'marital_status': 'MARITAL',
        'income': 'INCOME',
        'ideology': 'POLIDEOLOGY',
        'party_id': 'POLPARTY',
        'ethnicity': 'RACE'
    }
    
    # Process data to get mapped values
    actual_values = defaultdict(set)
    for instance in dataset:
        demo = instance['demographic']
        mapped_demo = {
            'age_group': map_age_group(demo.get('age')),
            'region': map_location_to_region(demo.get('location')),
            'gender': map_gender(demo.get('gender')),
            'education': map_education(demo.get('education')),
            'marital_status': map_marital_status(demo.get('marital_status')),
            'income': map_income(demo.get('income')),
            'ideology': map_ideology(demo.get('ideology')),
            'party_id': map_party_id(demo.get('party_id')),
            'ethnicity': map_ethnicity_to_race(demo.get('ethnicity'))
        }
        
        for field, target_key in field_map.items():
            value = mapped_demo.get(field)
            if value is not None:
                actual_values[target_key].add(value)
    
    print(f"\n{'='*60}")
    print("MAPPING VERIFICATION")
    print(f"{'='*60}")
    
    all_match = True
    for key in sorted(target_values.keys()):
        target = target_values[key]
        actual = actual_values.get(key, set())
        
        print(f"\n{key}:")
        print(f"  Target values ({len(target)}): {sorted(target)}")
        print(f"  Actual values ({len(actual)}): {sorted(actual)}")
        
        extra = actual - target
        missing = target - actual
        
        if extra:
            print(f"  ⚠️  EXTRA values (not in target): {sorted(extra)}")
            all_match = False
        if missing:
            print(f"  ℹ️  MISSING values (in target but not found in data): {sorted(missing)}")
        if not extra and not missing and actual:
            print(f"  ✓ All values match target!")
    
    if all_match:
        print(f"\n{'='*60}")
        print("✓ ALL MAPPINGS MATCH TARGET FORMAT!")
        print(f"{'='*60}")
    else:
        print(f"\n{'='*60}")
        print("⚠️  SOME MAPPINGS DO NOT MATCH - See details above")
        print(f"{'='*60}")
    
    return all_match

def verify_dataset_specific_mappings():
    """Verify all specific values from your dataset are correctly mapped"""
    
    print(f"\n{'='*60}")
    print("DATASET-SPECIFIC MAPPING VERIFICATION")
    print(f"{'='*60}")
    
    # Test party_id mappings
    print("\n1. PARTY_ID Mappings:")
    party_values = [
        'Moderate Republican',
        'Lean Democrat',
        "Don't Lean/Independent/None",
        'Moderate Democrat',
        'Strong Republican',
        'Lean Republican',
        'Strong Democrat'
    ]
    
    party_expected = {
        'Moderate Republican': 'Republican',
        'Lean Democrat': 'Democrat',
        "Don't Lean/Independent/None": 'Independent',
        'Moderate Democrat': 'Democrat',
        'Strong Republican': 'Republican',
        'Lean Republican': 'Republican',
        'Strong Democrat': 'Democrat'
    }
    
    all_correct = True
    for val in party_values:
        result = map_party_id(val)
        expected = party_expected[val]
        status = "✓" if result == expected else "✗"
        if result != expected:
            all_correct = False
        print(f"  {status} '{val}' → '{result}' (expected: '{expected}')")
    
    # Test education mappings
    print("\n2. EDUCATION Mappings:")
    education_values = [
        "Bachelor's degree",
        'High school graduate or equivalent',
        'Less than high school',
        'Vocational/tech school/some college/associates',
        'Post grad study/professional degree'
    ]
    
    education_expected = {
        "Bachelor's degree": 'college graduate/some postgrad',
        'High school graduate or equivalent': 'high school graduate',
        'Less than high school': 'less than high school',
        'Vocational/tech school/some college/associates': 'some college, no degree',
        'Post grad study/professional degree': 'postgraduate'
    }
    
    for val in education_values:
        result = map_education(val)
        expected = education_expected[val]
        status = "✓" if result == expected else "✗"
        if result != expected:
            all_correct = False
        print(f"  {status} '{val}' → '{result}' (expected: '{expected}')")
    
    # Test gender mappings
    print("\n3. GENDER Mappings:")
    gender_values = ['Female', 'Male']
    gender_expected = {'Female': 'female', 'Male': 'male'}
    
    for val in gender_values:
        result = map_gender(val)
        expected = gender_expected[val]
        status = "✓" if result == expected else "✗"
        if result != expected:
            all_correct = False
        print(f"  {status} '{val}' → '{result}' (expected: '{expected}')")
    
    # Test marital_status mappings
    print("\n4. MARITAL_STATUS Mappings:")
    marital_values = [
        'Married',
        'Widowed',
        'Separated',
        'Divorced',
        'Never married',
        'Living with partner'
    ]
    
    marital_expected = {
        'Married': 'married',
        'Widowed': 'widowed',
        'Separated': 'divorced',
        'Divorced': 'divorced',
        'Never married': 'never been married',
        'Living with partner': 'living with a partner'
    }
    
    for val in marital_values:
        result = map_marital_status(val)
        expected = marital_expected[val]
        status = "✓" if result == expected else "✗"
        if result != expected:
            all_correct = False
        print(f"  {status} '{val}' → '{result}' (expected: '{expected}')")
    
    # Test income mappings
    print("\n5. INCOME Mappings:")
    income_values = [
        '50-74K', '100-124K', '40-49K', '75-99K', '200K+',
        '30-39K', '20-29K', '175-200K+', '125-149K', '150-175K+'
    ]
    
    income_expected = {
        '50-74K': '$50,000-$75,000',
        '100-124K': '$100,000 or more',
        '40-49K': '$30,000-$50,000',
        '75-99K': '$75,000-$100,000',
        '200K+': '$100,000 or more',
        '30-39K': '$30,000-$50,000',
        '20-29K': 'Less than $30,000',
        '175-200K+': '$100,000 or more',
        '125-149K': '$100,000 or more',
        '150-175K+': '$100,000 or more'
    }
    
    for val in income_values:
        result = map_income(val)
        expected = income_expected[val]
        status = "✓" if result == expected else "✗"
        if result != expected:
            all_correct = False
        print(f"  {status} '{val}' → '{result}' (expected: '{expected}')")
    
    # Test location/region mappings (sample)
    print("\n6. LOCATION → REGION Mappings (sample):")
    location_samples = [
        ('Connecticut', 'Northeast'),
        ('California', 'West'),
        ('Tennessee', 'South'),
        ('Indiana', 'Midwest'),
        ('Alaska', 'West')
    ]
    
    for location, expected_region in location_samples:
        result = map_location_to_region(location)
        status = "✓" if result == expected_region else "✗"
        if result != expected_region:
            all_correct = False
        print(f"  {status} '{location}' → '{result}' (expected: '{expected_region}')")
    
    # Test age mappings (sample)
    print("\n7. AGE → AGE_GROUP Mappings (sample):")
    age_samples = [
        (18, '18-29'),
        (25, '18-29'),
        (30, '30-49'),
        (45, '30-49'),
        (50, '50-64'),
        (64, '50-64'),
        (65, '65+'),
        (94, '65+')
    ]
    
    for age, expected_group in age_samples:
        result = map_age_group(age)
        status = "✓" if result == expected_group else "✗"
        if result != expected_group:
            all_correct = False
        print(f"  {status} {age} → '{result}' (expected: '{expected_group}')")
    
    # Summary
    print(f"\n{'='*60}")
    if all_correct:
        print("✓ ALL DATASET-SPECIFIC VALUES MAPPED CORRECTLY!")
    else:
        print("✗ SOME MAPPINGS FAILED - Check details above")
    print(f"{'='*60}")
    
    return all_correct

# ============================================================================
# MAIN EXECUTION
# ============================================================================


In [ ]:

verify_dataset_specific_mappings()

# Load your dataset
print("\n" + "="*60)
print("STEP 1: Loading dataset...")
print("="*60)

unmapped = print_unmapped_values(data)

# Step 3: Verify mappings match target format
print("\n" + "="*60)
print("STEP 3: Verifying all mappings...")
print("="*60)
verify_mapping_values(data)

# Step 4: Process dataset - Single feature grouping
print("\n" + "="*60)
print("STEP 4: Processing dataset...")
print("="*60)

demographic_features = ['age', 'region', 'gender', 'education', 'marital_status', 
                        'income', 'party_id']

results, filtered_out = process_dataset(
    data, 
    demographic_features,
    min_group_size=100,  # Adjust as needed
    max_group_size=None
)


In [ ]:
len(results)

### Mapping Check

In [ ]:
results[0]

In [ ]:
target_variables=[]
for d in opinion[20:]:
    target_variables.append(str(d['group_prompt_variable_map']))

In [ ]:
for d in results:
    if len(d['group_prompt_variable_map']):
        assert(str(d['group_prompt_variable_map']) in target_variables)

In [ ]:
len(results)

In [ ]:
uni_demo=[]
for d in results:
    if d['group_prompt_variable_map']:
        uni_demo.append(str(list(d['group_prompt_variable_map'].keys())[0]))
print(len(set(uni_demo)))

In [ ]:
print(set(uni_demo))

In [ ]:
results[60]

In [ ]:
results[-30]

In [ ]:
results[0]

In [ ]:
with open('./dataset/socsci/grouped/grouped_socsci_{}.json'.format(data_name), 'w') as f:
    json.dump(results, f)

In [ ]:
sorted_id[idx][0]

### Dataset samples

In [ ]:
import csv

In [ ]:
with open (f'./dataset/socsci/grouped/grouped_socsci_{data_name}.json') as f:
    data=json.load(f)
with open(f'./dataset/socsci/socsci_{data_name}.json') as f:
    raw=json.load(f)

In [ ]:
uni_qs=[]
uni_demo=[]
uni_res=[]

for d in raw:
    uni_res.append(d['participant'])

for d in data:
    uni_qs.append(d['input_template'])
    uni_demo.append(str(d['group_prompt_variable_map']))
uni_rcnt=len(set(uni_res))
uni_qcnt=len(set(uni_qs))
uni_democnt=len(set(uni_demo))
print(uni_rcnt)
print(uni_qcnt)
print(uni_democnt)

In [ ]:
data_name

In [ ]:
csv_header = [[f'Study_id: {data_name}']]

csv_header.append([f'Number of unique respondent: {uni_rcnt}'])
csv_header.append([f'AVG survey response per respondent: {round(len(raw)/uni_rcnt ,2)}'])

csv_header.append([f'Unique number of questions: {uni_qcnt}'])
csv_header.append([f'Unique number of demographic groups: {uni_democnt}'])
csv_header.append([f'Total number of instanes {len(data)}'])

In [ ]:
for d in data:
    if len(d['group_prompt_variable_map'])==0:
        csv_header.append([d['input_template']])

In [ ]:
with open(f'./dataset/socsci/grouped/{data_name}_samples.csv', 'w', newline="") as f:
    writer = csv.writer(f)
    writer.writerows(csv_header)


### Group per individual

In [ ]:
import json
data_name='mp3dr'
with open(f'./dataset/socsci/socsci_{data_name}.json') as f:
    data=json.load(f)

In [ ]:
data[0]

In [ ]:
individuals={}
for d in data:
    if d['participant'] not in individuals.keys():
        individuals[d['participant']]=[]
        individuals[d['participant']].append(d)
    else:
        individuals[d['participant']].append(d)

save_data=[]
for key in individuals.keys():
    save_data.append(individuals[key])

In [ ]:
with open(f'./dataset/socsci/grouped/socsci_{data_name}_per_respondent.json', 'w') as f:
    json.dump(save_data, f)

In [ ]:
len(data)

In [ ]:
with open('./dataset/socsci/grouped/socsci_nb4xg_per_respondent.json') as f:
    target=json.load(f)

In [ ]:
target[0]

In [ ]:
data[0]

In [ ]:
len(grouped_results)

In [ ]:
uni_qs=[]
for d in data:
    uni_qs.append(d['stimuli'])
print(len(set(uni_qs)))

In [ ]:
data[0]

In [ ]:
data[9]

In [ ]:
grouped_results[4]

In [ ]:
grouped_results[0]

In [ ]:
data[8]

In [ ]:
sorted_id[0]

In [ ]:
print(set(studies))

In [ ]:
print('hi')

### Statistics compared to OpinionQA

In [ ]:
for key in keys_val.keys():
    print(f"Key: {key}")
    print(Counter(keys_val[key]))
    print()

In [ ]:
# Define income order
income_order = ['<5K', '5-9K', '10-14K', '15-19K', '20-29K', '30-39K', '40-49K', 
                '50-74K', '75-99K', '100-124K', '125-149K', '150-175K+', '175-200K+', '200K+']

for key in keys_val.keys():
    counter = Counter(keys_val[key])
    
    # Remove None
    counter.pop(None, None)
    
    # Sort appropriately
    if key == 'age':
        # Sort numerically in ascending order
        sorted_items = sorted(counter.items(), key=lambda x: x[0])
    elif key == 'income':
        # Sort by income range order
        sorted_items = [(inc, counter[inc]) for inc in income_order if inc in counter]
    elif key == 'household_size':
        # Sort numerically
        sorted_items = sorted(counter.items(), key=lambda x: x[0])
    else:
        # Get top 20 most common
        sorted_items = counter.most_common(20)
    
    labels, values = zip(*sorted_items) if sorted_items else ([], [])
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(labels)), values, color='steelblue', alpha=0.8)
    
    # Add value labels on each bar
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.xticks(range(len(labels)), [str(l) for l in labels], rotation=45, ha='right')
    plt.title(f'{key.replace("_", " ").title()} Distribution', fontsize=14)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig(f'{key}_distribution.png', dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
# Define region mappings for US states
region_mapping = {
    # Northeast
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast',
    'New Hampshire': 'Northeast', 'Rhode Island': 'Northeast', 'Vermont': 'Northeast',
    'New Jersey': 'Northeast', 'New York': 'Northeast', 'Pennsylvania': 'Northeast',
    
    # Midwest
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest',
    'Michigan': 'Midwest', 'Minnesota': 'Midwest', 'Missouri': 'Midwest', 'Nebraska': 'Midwest',
    'North Dakota': 'Midwest', 'Ohio': 'Midwest', 'South Dakota': 'Midwest', 'Wisconsin': 'Midwest',
    
    # South
    'Alabama': 'South', 'Arkansas': 'South', 'Delaware': 'South', 'Florida': 'South',
    'Georgia': 'South', 'Kentucky': 'South', 'Louisiana': 'South', 'Maryland': 'South',
    'Mississippi': 'South', 'North Carolina': 'South', 'Oklahoma': 'South', 'South Carolina': 'South',
    'Tennessee': 'South', 'Texas': 'South', 'Virginia': 'South', 'West Virginia': 'South',
    'District of Columbia': 'South',
    
    # West
    'Alaska': 'West', 'Arizona': 'West', 'California': 'West', 'Colorado': 'West',
    'Hawaii': 'West', 'Idaho': 'West', 'Montana': 'West', 'Nevada': 'West',
    'New Mexico': 'West', 'Oregon': 'West', 'Utah': 'West', 'Washington': 'West', 'Wyoming': 'West'
}

# Age mappings
def map_age(age):
    if age is None:
        return None
    if 18 <= age <= 29:
        return '18-29'
    elif 30 <= age <= 49:
        return '30-49'
    elif 50 <= age <= 64:
        return '50-64'
    elif age >= 65:
        return '65+'
    return None

# Gender mappings
gender_mapping = {
    'Male': 'male',
    'Female': 'female'
}

# Education mappings
education_mapping = {
    'Less than high school': 'less than high school',
    'Some high school (no diploma)': 'less than high school',
    'High school graduate or equivalent': 'high school graduate',
    'Vocational/tech school/some college/associates': 'some college, no degree',
    "Bachelor's degree": 'college graduate/some postgrad',
    'Post grad study/professional degree': 'postgraduate'
}

# Marital status mappings
marital_mapping = {
    'Married': 'married',
    'Never married': 'never been married',
    'Divorced': 'divorced',
    'Living with partner': 'living with a partner',
    'Widowed': 'widowed',
    'Separated': 'divorced'  # Mapping separated to divorced
}

# Party ID mappings
party_mapping = {
    'Strong Democrat': 'Democrat',
    'Moderate Democrat': 'Democrat',
    'Lean Democrat': 'Democrat',
    'Strong Republican': 'Republican',
    'Moderate Republican': 'Republican',
    'Lean Republican': 'Republican',
    "Don't Lean/Independent/None": 'Independent'
}

# Income mappings
income_mapping = {
    '<5K': 'Less than $30,000',
    '5-9K': 'Less than $30,000',
    '10-14K': 'Less than $30,000',
    '15-19K': 'Less than $30,000',
    '20-29K': 'Less than $30,000',
    '30-39K': '$30,000-$50,000',
    '40-49K': '$30,000-$50,000',
    '50-74K': '$50,000-$75,000',
    '75-99K': '$75,000-$100,000',
    '100-124K': '$100,000 or more',
    '125-149K': '$100,000 or more',
    '150-175K+': '$100,000 or more',
    '175-200K+': '$100,000 or more',
    '200K+': '$100,000 or more'
}

# Ideology mappings
ideology_mapping = {
    'Very Liberal': 'very liberal',
    'Liberal': 'liberal',
    'Extremely Liberal': 'very liberal',
    'Moderate': 'moderate',
    'Conservative': 'conservative',
    'Somewhat Conservative': 'conservative',
    'Very Conservative': 'very conservative',
    'Extremely Conservative': 'very conservative',
    'Somewhat Liberal': 'liberal'
}

# Race/Ethnicity mappings
ethnicity_mapping = {
    'White': 'white',
    'Black': 'black',
    'Hispanic': 'hispanic',
    'Asian': 'asian'
}

# Initialize data containers for each feature
data_splits = {
    'cregion': defaultdict(list),
    'age': defaultdict(list),
    'sex': defaultdict(list),
    'education': defaultdict(list),
    'marital': defaultdict(list),
    'polparty': defaultdict(list),
    'income': defaultdict(list),
    'polideology': defaultdict(list),
    'race': defaultdict(list)
}

### Split data based on mapping with OpinionQA

In [ ]:
import json
from collections import defaultdict
from tqdm import tqdm
import re

# [Include all the mapping dictionaries from above]

# ... (same mappings as above)

# Process dataset and save files
print("Processing dataset...")

def clean_filename(text):
    """Convert category name to clean filename format"""
    # Remove special characters, convert to lowercase, replace spaces with underscores
    text = text.lower()
    text = text.replace('$', '').replace(',', '').replace('+', 'plus')
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'[-\s]+', '_', text)
    return text

data_splits = {
    'cregion': defaultdict(list),
    'age': defaultdict(list),
    'sex': defaultdict(list),
    'education': defaultdict(list),
    'marital': defaultdict(list),
    'polparty': defaultdict(list),
    'income': defaultdict(list),
    'polideology': defaultdict(list),
    'race': defaultdict(list)
}

for d in tqdm(dataset['train']):
    demo = d['demographic']
    
    # CREGION (location)
    if demo.get('location') and demo['location'] in region_mapping:
        region = region_mapping[demo['location']]
        data_splits['cregion'][region.lower()].append(d)
    
    # AGE
    if demo.get('age'):
        age_group = map_age(demo['age'])
        if age_group:
            data_splits['age'][age_group].append(d)
    
    # SEX (gender)
    if demo.get('gender') and demo['gender'] in gender_mapping:
        sex = gender_mapping[demo['gender']]
        data_splits['sex'][sex].append(d)
    
    # EDUCATION
    if demo.get('education') and demo['education'] in education_mapping:
        edu = education_mapping[demo['education']]
        data_splits['education'][edu].append(d)
    
    # MARITAL
    if demo.get('marital_status') and demo['marital_status'] in marital_mapping:
        marital = marital_mapping[demo['marital_status']]
        data_splits['marital'][marital].append(d)
    
    # POLPARTY (party_id)
    if demo.get('party_id') and demo['party_id'] in party_mapping:
        party = party_mapping[demo['party_id']]
        data_splits['polparty'][party.lower()].append(d)
    
    # INCOME
    if demo.get('income') and demo['income'] in income_mapping:
        inc = income_mapping[demo['income']]
        data_splits['income'][inc].append(d)
    
    # POLIDEOLOGY (ideology)
    if demo.get('ideology') and demo['ideology'] in ideology_mapping:
        ideo = ideology_mapping[demo['ideology']]
        data_splits['polideology'][ideo].append(d)
    
    # RACE (ethnicity)
    if demo.get('ethnicity') and demo['ethnicity'] in ethnicity_mapping:
        race = ethnicity_mapping[demo['ethnicity']]
        data_splits['race'][race].append(d)

# Save to JSON files
print("\nSaving files...")
for feature, categories in data_splits.items():
    for category, data in categories.items():
        category_clean = clean_filename(category)
        filename = f"socsci_{feature}_{category_clean}.json"
        
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        
        print(f"Saved {filename} with {len(data)} instances")

# Print summary
print("\n=== Summary ===")
for feature, categories in data_splits.items():
    print(f"\n{feature.upper()}:")
    total = 0
    for category, data in sorted(categories.items()):
        print(f"  {category}: {len(data)} instances")
        total += len(data)
    print(f"  Total: {total}")


In [ ]:
dataset['train'][1]

In [ ]:
save_data=[]
for d in dataset['train']:
    save_data.append(d)


In [ ]:
with open('./dataset/socsci/socsci.json', 'w') as f:
    json.dump()

In [ ]:
participant_mapping